# Weight Initialization
In this lesson, you'll learn how to find good initial weights for a neural network. Having good initial weights can place the neural network close to the optimal solution. This allows the neural network to come to the best solution quicker. 

## Testing Weights
### Dataset
To see how different weights perform, we'll test on the same dataset and neural network. Let's go over the dataset and neural network.

We'll be using the [MNIST dataset](https://en.wikipedia.org/wiki/MNIST_database) to demonstrate the different initial weights. As a reminder, the MNIST dataset contains images of handwritten numbers, 0-9, with normalized input (0.0 - 1.0).  Run the cell below to download and load the MNIST dataset.

In [ ]:
%matplotlib inline

import numpy as np
import torch
import helper

print('Getting MNIST Dataset...')
mnist = helper.load_mnist()
print('Data Extracted.')

### Neural Network
<img style="float: left" src="images/neural_network.png"/>

For the neural network, we'll test on a 3 layer neural network with ReLU activations and an Adam optimizer.  The lessons you learn apply to other neural networks, including different activations and optimizers.

The weights are plain tensors (shape `(inputs, outputs)`) that the helper trains for us, so we can focus on how they are initialized.

In [ ]:
# Save the shapes of weights for each layer
layer_1_weight_shape = (mnist.train.images.shape[1], 256)
layer_2_weight_shape = (256, 128)
layer_3_weight_shape = (128, mnist.train.labels.shape[1])

## Initialize Weights
Let's start looking at some initial weights.
### All Zeros or Ones
If you follow the principle of [Occam's razor](https://en.wikipedia.org/wiki/Occam's_razor), you might think setting all the weights to 0 or 1 would be the best solution.  This is not the case.

With every weight the same, all the neurons at each layer are producing the same output.  This makes it hard to decide which weights to adjust.

Let's compare the loss with all ones and all zero weights using `helper.compare_init_weights`.  This function will run two different initial weights on the neural network above for 2 epochs.  It will plot the loss for the first 100 batches and print out stats after the 2 epochs (~860 batches). We plot the first 100 batches to better judge which weights performed better at the start.

Run the cell below to see the difference between weights of all zeros against all ones.

In [ ]:
all_zero_weights = [
    torch.zeros(layer_1_weight_shape),
    torch.zeros(layer_2_weight_shape),
    torch.zeros(layer_3_weight_shape)
]

all_one_weights = [
    torch.ones(layer_1_weight_shape),
    torch.ones(layer_2_weight_shape),
    torch.ones(layer_3_weight_shape)
]

helper.compare_init_weights(
    mnist,
    'All Zeros vs All Ones',
    [
        (all_zero_weights, 'All Zeros'),
        (all_one_weights, 'All Ones')])

As you can see the accuracy is close to guessing for both zeros and ones, around 10%.

The neural network is having a hard time determining which weights need to be changed, since the neurons have the same output for each layer.  To avoid neurons with the same output, let's use unique weights.  We can also randomly select these weights to avoid being stuck in a local minimum for each run.

A good solution for getting these random weights is to sample from a uniform distribution.

### Uniform Distribution
A [uniform distribution](https://en.wikipedia.org/wiki/Uniform_distribution_(continuous%29) has the equal probability of picking any number from a set of numbers. We'll be picking from a continous distribution, so the chance of picking the same number is low. We'll use PyTorch's `uniform_` method to pick random numbers from a uniform distribution.

>#### [`Tensor.uniform_(from=0, to=1)`](https://pytorch.org/docs/stable/generated/torch.Tensor.uniform_.html)
>Fills the tensor in place with random values from a uniform distribution.

>The generated values follow a uniform distribution in the range [from, to). The lower bound `from` is included in the range, while the upper bound `to` is excluded.

>- **from:** The lower bound on the range of random values to generate. Defaults to 0.
- **to:** The upper bound on the range of random values to generate. Defaults to 1.

>`torch.empty(shape)` creates an uninitialized tensor of the given shape for `uniform_` to fill, and `torch.rand(shape)` is a shortcut for `from=0, to=1`.

We can visualize the uniform distribution by using a histogram. Let's map the values from `torch.empty(1000).uniform_(-3, 3)` to a histogram using the `helper.hist_dist` function. This will be `1000` random float values from `-3` to `3`, excluding the value `3`.

In [ ]:
helper.hist_dist('Random Uniform (minval=-3, maxval=3)', torch.empty(1000).uniform_(-3, 3))

The histogram used 500 buckets for the 1000 values.  Since the chance for any single bucket is the same, there should be around 2 values for each bucket. That's exactly what we see with the histogram.  Some buckets have more and some have less, but they trend around 2.

Now that you understand how to draw uniform random numbers, let's apply it to some initial weights.

### Baseline


Let's see how well the neural network trains using the default range for `torch.rand`, which is `[0.0, 1.0)`.

In [ ]:
# torch.rand draws from [0, 1)
basline_weights = [
    torch.rand(layer_1_weight_shape),
    torch.rand(layer_2_weight_shape),
    torch.rand(layer_3_weight_shape)
]

helper.compare_init_weights(
    mnist,
    'Baseline',
    [(basline_weights, 'torch.rand [0, 1)')])

The loss graph is showing the neural network is learning, which it didn't with all zeros or all ones. We're headed in the right direction.

### General rule for setting weights
The general rule for setting the weights in a neural network is to be close to zero without being too small. A good practice is to start your weights in the range of $[-y, y]$ where
$y=1/\sqrt{n}$ ($n$ is the number of inputs to a given neuron).

Let's see if this holds true, let's first center our range over zero.  This will give us the range [-1, 1).

In [ ]:
uniform_neg1to1_weights = [
    torch.empty(layer_1_weight_shape).uniform_(-1, 1),
    torch.empty(layer_2_weight_shape).uniform_(-1, 1),
    torch.empty(layer_3_weight_shape).uniform_(-1, 1)
]

helper.compare_init_weights(
    mnist,
    '[0, 1) vs [-1, 1)',
    [
        (basline_weights, 'torch.rand [0, 1)'),
        (uniform_neg1to1_weights, 'uniform [-1, 1)')])

We're going in the right direction, the accuracy and loss is better with [-1, 1). We still want smaller weights. How far can we go before it's too small?

### Too small
Let's compare [-0.1, 0.1), [-0.01, 0.01), and [-0.001, 0.001) to see how small is too small.  We'll also set `plot_n_batches=None` to show all the batches in the plot.

In [ ]:
uniform_neg01to01_weights = [
    torch.empty(layer_1_weight_shape).uniform_(-0.1, 0.1),
    torch.empty(layer_2_weight_shape).uniform_(-0.1, 0.1),
    torch.empty(layer_3_weight_shape).uniform_(-0.1, 0.1)
]

uniform_neg001to001_weights = [
    torch.empty(layer_1_weight_shape).uniform_(-0.01, 0.01),
    torch.empty(layer_2_weight_shape).uniform_(-0.01, 0.01),
    torch.empty(layer_3_weight_shape).uniform_(-0.01, 0.01)
]

uniform_neg0001to0001_weights = [
    torch.empty(layer_1_weight_shape).uniform_(-0.001, 0.001),
    torch.empty(layer_2_weight_shape).uniform_(-0.001, 0.001),
    torch.empty(layer_3_weight_shape).uniform_(-0.001, 0.001)
]

helper.compare_init_weights(
    mnist,
    '[-1, 1) vs [-0.1, 0.1) vs [-0.01, 0.01) vs [-0.001, 0.001)',
    [
        (uniform_neg1to1_weights, '[-1, 1)'),
        (uniform_neg01to01_weights, '[-0.1, 0.1)'),
        (uniform_neg001to001_weights, '[-0.01, 0.01)'),
        (uniform_neg0001to0001_weights, '[-0.001, 0.001)')],
    plot_n_batches=None)

Looks like anything [-0.01, 0.01) or smaller is too small.  Let's compare this to our typical rule of using the range $y=1/\sqrt{n}$.

In [ ]:
general_rule_weights = [
    torch.empty(layer_1_weight_shape).uniform_(-1/np.sqrt(layer_1_weight_shape[0]), 1/np.sqrt(layer_1_weight_shape[0])),
    torch.empty(layer_2_weight_shape).uniform_(-1/np.sqrt(layer_2_weight_shape[0]), 1/np.sqrt(layer_2_weight_shape[0])),
    torch.empty(layer_3_weight_shape).uniform_(-1/np.sqrt(layer_3_weight_shape[0]), 1/np.sqrt(layer_3_weight_shape[0]))
]

helper.compare_init_weights(
    mnist,
    '[-0.1, 0.1) vs General Rule',
    [
        (uniform_neg01to01_weights, '[-0.1, 0.1)'),
        (general_rule_weights, 'General Rule')],
    plot_n_batches=None)

The range we found and $y=1/\sqrt{n}$ are really close.

Since the uniform distribution has the same chance to pick anything in the range, what if we used a distribution that had a higher chance of picking numbers closer to 0.  Let's look at the normal distribution.
### Normal Distribution
Unlike the uniform distribution, the [normal distribution](https://en.wikipedia.org/wiki/Normal_distribution) has a higher likelihood of picking number close to it's mean. To visualize it, let's plot values from PyTorch's `torch.randn` function to a histogram.

>[torch.randn(*size)](https://pytorch.org/docs/stable/generated/torch.randn.html)

>Returns a tensor filled with random numbers from a normal distribution with mean 0 and standard deviation 1.

>- **size:** The shape of the output tensor.

>To get a different mean and standard deviation, scale and shift the result: `mean + stddev * torch.randn(size)`, or use `Tensor.normal_(mean, std)` to fill an existing tensor.

In [ ]:
helper.hist_dist('Random Normal (mean=0.0, stddev=1.0)', torch.randn(1000))

Let's compare the normal distribution against the previous uniform distribution.

In [ ]:
normal_01_weights = [
    torch.randn(layer_1_weight_shape) * 0.1,
    torch.randn(layer_2_weight_shape) * 0.1,
    torch.randn(layer_3_weight_shape) * 0.1
]

helper.compare_init_weights(
    mnist,
    'Uniform [-0.1, 0.1) vs Normal stddev 0.1',
    [
        (uniform_neg01to01_weights, 'Uniform [-0.1, 0.1)'),
        (normal_01_weights, 'Normal stddev 0.1')])

The normal distribution gave a slight increase in accuracy and loss.  Let's move closer to 0 and drop picked numbers that are `x` number of standard deviations away.  This distribution is called [Truncated Normal Distribution](https://en.wikipedia.org/wiki/Truncated_normal_distribution%29).
### Truncated Normal Distribution
>[torch.nn.init.trunc_normal_(tensor, mean=0.0, std=1.0, a=-2.0, b=2.0)](https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.trunc_normal_)

>Fills the tensor in place with values from a truncated normal distribution.

>The generated values follow a normal distribution with specified mean and standard deviation, except that values outside `[a, b]` are redrawn until they fall inside it.

>- **tensor:** The tensor to fill.
- **mean:** The mean of the normal distribution.
- **std:** The standard deviation of the normal distribution.
- **a, b:** The minimum and maximum cutoff values. Note these are absolute values, not multiples of `std`.

We want the same behavior as the classic truncated normal: drop values more than 2 standard deviations from the mean. So we set `a=-2*std` and `b=2*std`. The helper function below does that.

In [ ]:
def truncated_normal(shape, stddev=1.0):
    """Normal values, but any value more than 2 standard deviations from 0 is re-picked."""
    return torch.nn.init.trunc_normal_(torch.empty(shape), std=stddev, a=-2*stddev, b=2*stddev)

helper.hist_dist('Truncated Normal (mean=0.0, stddev=1.0)', truncated_normal([1000]))

Again, let's compare the previous results with the previous distribution.

In [ ]:
trunc_normal_01_weights = [
    truncated_normal(layer_1_weight_shape, stddev=0.1),
    truncated_normal(layer_2_weight_shape, stddev=0.1),
    truncated_normal(layer_3_weight_shape, stddev=0.1)
]

helper.compare_init_weights(
    mnist,
    'Normal vs Truncated Normal',
    [
        (normal_01_weights, 'Normal'),
        (trunc_normal_01_weights, 'Truncated Normal')])

There's no difference between the two, but that's because the neural network we're using is too small. A larger neural network will pick more points on the normal distribution, increasing the likelihood it's choices are larger than 2 standard deviations.

We've come a long way from the first set of weights we tested. Let's see the difference between the weights we used then and now.

In [ ]:
helper.compare_init_weights(
    mnist,
    'Baseline vs Truncated Normal',
    [
        (basline_weights, 'Baseline'),
        (trunc_normal_01_weights, 'Truncated Normal')])

That's a huge difference. You can barely see the truncated normal line.  However, this is not the end your learning path.  We've provided more resources for initializing weights in the classroom!